In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader,random_split
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

##Dataset
class MyDataset(Dataset):
    def __init__(self):
        #生成两类数据，每类500条
        X0=torch.randn(500,2)+torch.tensor([2.0,2.0])
        X1=torch.randn(500,2)+torch.tensor([-2.0,-2.0])
        self.X=torch.cat([X0,X1],dim=0)
        self.y=torch.cat([
            torch.zeros(500,1),
            torch.ones(500,1)
        ],dim=0)

    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx):
        #idx是PyTorch传进来的索引
        return self.X[idx],self.y[idx]

dataset=MyDataset()

#划分训练集/验证集（8:2）
train_size=int(0.8*len(dataset))
val_size=len(dataset)-train_size
train_set,val_set=random_split(dataset,[train_size,val_size])

train_loader=DataLoader(train_set,batch_size=32,shuffle=True)
val_loader=DataLoader(val_set,batch_size=32,shuffle=False)

print(f"总样本:{len(dataset)},训练集:{len(train_set)},验证集:{len(val_set)}")

        

总样本:1000,训练集:800,验证集:200


In [2]:
class MLP(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        #第一层：2维 -> 64维
        self.layer1=nn.Linear(input_dim,64)
        self.relu1=nn.ReLU()
        self.dropout1=nn.Dropout(0.3)  #训练时随机关掉30%神经元

        #第二层：64维 -> 32维
        self.layer2=nn.Linear(64,32)
        self.relu2=nn.ReLU()
        self.dropout2=nn.Dropout(0.3)

        #第三层：32维 -> 1维（输出）
        self.layer3=nn.Linear(32,1)
        self.sigmoid=nn.Sigmoid()

    def forward(self,x):
        x=self.layer1(x)
        x=self.relu1(x)
        x=self.dropout1(x)   #只在训练时生效，预测时自动关闭

        x=self.layer2(x)
        x=self.relu2(x)
        x=self.dropout2(x)

        x=self.layer3(x)
        return self.sigmoid(x)

#创建模型
model=MLP(input_dim=2)
print(model)


MLP(
  (layer1): Linear(in_features=2, out_features=64, bias=True)
  (relu1): ReLU()
  (dropout1): Dropout(p=0.3, inplace=False)
  (layer2): Linear(in_features=64, out_features=32, bias=True)
  (relu2): ReLU()
  (dropout2): Dropout(p=0.3, inplace=False)
  (layer3): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [3]:
criterion=nn.BCELoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.01)  #Adam比SGD更稳

epochs=100

for epoch in range(epochs):
    #训练模式
    model.train()  #开启Dropout
    train_loss=0
    train_acc=0

    for x_batch,y_batch in train_loader:
        y_pred=model(x_batch)
        loss=criterion(y_pred,y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss+=loss.item()
        acc=((y_pred>=0.5).float()==y_batch).float().mean()
        train_acc+=acc.item()

    #验证模式
    model.eval()  #关闭Dropout
    val_loss=0
    val_acc=0

    with torch.no_grad():    #验证时不求导，省内存
        for x_batch,y_batch in val_loader:
            y_pred=model(x_batch)
            loss=criterion(y_pred,y_batch)
            val_loss+=loss.item()
            acc=((y_pred>=0.5).float()==y_batch).float().mean()
            val_acc+=acc.item()

    #打印
    if epoch %10==0:
        print(f"Epoch{epoch:3d}|"
              f"Train Loss={train_loss/len(train_loader):.4f},Acc={train_acc/len(train_loader):.4f} |"
              f"Val Loss={val_loss/len(val_loader):.4f},Acc={val_acc/len(val_loader):.4f}")

print("\n训练完成！")

Epoch  0|Train Loss=0.0697,Acc=0.9862 |Val Loss=0.0016,Acc=1.0000
Epoch 10|Train Loss=0.0084,Acc=0.9962 |Val Loss=0.0009,Acc=1.0000
Epoch 20|Train Loss=0.0047,Acc=0.9988 |Val Loss=0.0015,Acc=1.0000
Epoch 30|Train Loss=0.0048,Acc=0.9962 |Val Loss=0.0007,Acc=1.0000
Epoch 40|Train Loss=0.0033,Acc=0.9975 |Val Loss=0.0002,Acc=1.0000
Epoch 50|Train Loss=0.0033,Acc=1.0000 |Val Loss=0.0006,Acc=1.0000
Epoch 60|Train Loss=0.0037,Acc=0.9975 |Val Loss=0.0004,Acc=1.0000
Epoch 70|Train Loss=0.0046,Acc=0.9962 |Val Loss=0.0021,Acc=1.0000
Epoch 80|Train Loss=0.0027,Acc=0.9988 |Val Loss=0.0001,Acc=1.0000
Epoch 90|Train Loss=0.0029,Acc=0.9988 |Val Loss=0.0000,Acc=1.0000

训练完成！


In [4]:
#最终验证
model.eval()
with torch.no_grad():
    X_all=dataset.X
    y_all=dataset.y
    y_prob=model(X_all)
    y_pred=(y_prob>=0.5).float()
    final_acc=(y_pred==y_all).float().mean()
    print(f"最终准确率:{final_acc.item():.2%}")

#保存模型
torch.save(model.state_dict(),"mlp_model.pth")
print("模型已保存到mlp_model.pth")

最终准确率:100.00%
模型已保存到mlp_model.pth


In [ ]:
print("a")